# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
# Task 1 — CO2 per capita, all countries, ONE highlighted
# Lecture 3 recipe: grey everything -> highlight one -> direct label -> insight title

GREY = '#EEEEEE'          # context lines (required by the exercise)
LABEL_GREY = '#AAAAAA'    # #EEEEEE is too pale to read as text, so the label uses a darker grey
HIGHLIGHT_COLOR = 'crimson'

def highlight_line_chart(data, value_col, highlight, y_label, title, height=450):
    """Grey context lines + one coloured, thicker, directly-labelled line."""
    countries = data['Country'].unique()

    # Colour map: highlight in colour, everything else grey (Lecture: Example 2)
    color_map = {c: HIGHLIGHT_COLOR if c == highlight else GREY for c in countries}

    # Draw the highlighted country LAST so grey lines never cover it
    plot_df = pd.concat([data[data['Country'] != highlight],
                         data[data['Country'] == highlight]])

    fig = px.line(plot_df, x='Year', y=value_col, color='Country',
                  color_discrete_map=color_map,
                  labels={value_col: y_label, 'Year': ''},
                  title=title,
                  template='plotly_white',      # no chart border / background clutter
                  height=height)

    # Thin context lines, no legend (direct labels replace it)
    fig.update_traces(line=dict(width=1), showlegend=False)
    # Thicker highlighted line
    fig.update_traces(line=dict(width=3), selector=dict(name=highlight))

    last_year = data['Year'].max()
    last = data[data['Year'] == last_year]

    # Direct label at the end of the highlighted line
    h_row = last[last['Country'] == highlight].iloc[0]
    fig.add_annotation(x=h_row['Year'], y=h_row[value_col], text=highlight,
                       showarrow=False, xanchor='left', xshift=10,
                       font=dict(color=HIGHLIGHT_COLOR, size=12, weight='bold', family='Arial'))

    # One "Other countries" label, placed at the end of the highest grey line
    o_row = last[last['Country'] != highlight].sort_values(value_col, ascending=False).iloc[0]
    fig.add_annotation(x=o_row['Year'], y=o_row[value_col], text='Other countries',
                       showarrow=False, xanchor='left', xshift=10,
                       font=dict(color=LABEL_GREY, size=12, weight='bold', family='Arial'))

    fig.update_layout(
        font=dict(size=13),
        margin=dict(r=130),                           # room for the end-of-line labels
        yaxis=dict(gridcolor='#EEEEEE', gridwidth=0.5),  # light gridlines
        xaxis=dict(showgrid=False, tickangle=0,          # no vertical grid, no rotated text
                   range=[data['Year'].min(), last_year]),
    )
    return fig


# ---- Choose the country to highlight ----
highlight = 'United Kingdom'

# ---- Build the insight title from the data (so the numbers are always correct) ----
h = df[df['Country'] == highlight].sort_values('Year')
start, end = h.iloc[0], h.iloc[-1]
pct = (end['CO2_per_capita'] / start['CO2_per_capita'] - 1) * 100
verb = 'rose' if pct >= 0 else 'fell'

# change in CO2 per capita for every country, to rank the highlighted one (1 = steepest fall)
first = df[df['Year'] == start['Year']].set_index('Country')['CO2_per_capita']
last = df[df['Year'] == end['Year']].set_index('Country')['CO2_per_capita']
change = (last / first - 1) * 100
rank = int(change.rank(ascending=True, method='min')[highlight])
rank_text = 'the steepest fall' if rank == 1 else f'fall ranked #{rank}'

title = (f"{highlight}'s CO2 per capita {verb} {abs(pct):.0f}% since {int(start['Year'])} "
         f"— {rank_text} of {len(change)} countries")

fig1 = highlight_line_chart(df, 'CO2_per_capita', highlight,
                            y_label='CO2 per capita', title=title)
fig1.show()


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [3]:
# Task 2 — CO2 emissions over time, Asian countries only, ONE highlighted
asia = df[df['Region'] == 'Asia']
print("Asian countries:", sorted(asia['Country'].unique()))

# ---- Choose the country to highlight ----
highlight = 'China'

# ---- Insight title built from the data ----
h = asia[asia['Country'] == highlight].sort_values('Year')
start, end = h.iloc[0], h.iloc[-1]
pct = (end['CO2_Mt'] / start['CO2_Mt'] - 1) * 100
verb = 'rose' if pct >= 0 else 'fell'

# growth rank among Asian countries (1 = fastest growing)
growth = (asia[asia['Year'] == end['Year']].set_index('Country')['CO2_Mt']
          / asia[asia['Year'] == start['Year']].set_index('Country')['CO2_Mt'] - 1)
g_rank = int(growth.rank(ascending=False, method='min')[highlight])
rank_text = ('the fastest growth' if g_rank == 1
             else f'growth ranked #{g_rank}')

title = (f"{highlight}'s CO2 emissions {verb} {abs(pct):.0f}% since {int(start['Year'])} "
         f"— {rank_text} among {asia['Country'].nunique()} Asian countries")

fig2 = highlight_line_chart(asia, 'CO2_Mt', highlight,
                            y_label='CO2 Emissions (Mt)', title=title)
fig2.show()
